# The Price Is Right - Week 7

## Day 3 and Day 4: Training!

This is what it's all been about!

If you are using LITE_MODE=True, then please run this on a free T4 box.

If you are using LITE_MODE-False, then please use a paid A100 with high memory.

In [2]:
!wget -q https://github.com/Abhishekravindran/LLM_Engineering_opensource/tree/main/finetuning_local_frontier_models/util.py -O util.py

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
          "GB")
else:
    raise RuntimeError("Please enable a GPU runtime.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    huggingface_hub \
    wandb \
    tqdm \
    matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curr

In [5]:
import os
import re
import math
from datetime import datetime

import torch
import transformers
import wandb
import matplotlib.pyplot as plt

from tqdm import tqdm
from datasets import load_dataset, Dataset, DatasetDict

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [6]:
base_model = "Qwen/Qwen3-0.6B"

In [7]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

PROJECT_NAME = "price"

LITE_MODE = True

# Your Hugging Face username
HF_USER = "Charan2804"

# Dataset
DATA_USER = "AbhishekRavindran"

DATASET_NAME = (
    f"{DATA_USER}/items_prompts_lite"
    if LITE_MODE
    else f"{DATA_USER}/items_prompts_full"
)

# Run name
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"

if LITE_MODE:
    RUN_NAME += "-lite"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# ============================================================
# TRAINING PARAMETERS
# ============================================================

EPOCHS = 1

# Start small for free Colab
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16

MAX_SEQUENCE_LENGTH = 128


# ============================================================
# QLoRA PARAMETERS
# ============================================================

QUANT_4_BIT = True

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]


# ============================================================
# OPTIMIZATION
# ============================================================

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
WEIGHT_DECAY = 0.001

OPTIMIZER = "paged_adamw_8bit"


# ============================================================
# EVALUATION / LOGGING
# ============================================================

VAL_SIZE = 500

LOG_STEPS = 5
SAVE_STEPS = 100

LOG_TO_WANDB = True


# ============================================================
# GPU PRECISION
# ============================================================

capability = torch.cuda.get_device_capability()

use_bf16 = capability[0] >= 8

print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", use_bf16)
print("Model:", base_model)
print("Dataset:", DATASET_NAME)

GPU: Tesla T4
BF16: False
Model: Qwen/Qwen3-0.6B
Dataset: AbhishekRavindran/items_prompts_lite


In [8]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")

if hf_token:
    login(
        token=hf_token,
        add_to_git_credential=True
    )
    print("✅ Hugging Face login successful")
else:
    print("ℹ️ HF_TOKEN not found. Continuing without Hugging Face login.")

✅ Hugging Face login successful


In [9]:
if LOG_TO_WANDB:

    wandb_api_key = userdata.get("WANDB_API_KEY")

    if wandb_api_key:
        os.environ["WANDB_API_KEY"] = wandb_api_key

        wandb.login()

        os.environ["WANDB_PROJECT"] = PROJECT_NAME
        os.environ["WANDB_LOG_MODEL"] = "false"
        os.environ["WANDB_WATCH"] = "false"

        print("✅ W&B login successful")

    else:
        print("⚠️ WANDB_API_KEY not found")
        LOG_TO_WANDB = False

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: charansathish455 (charan_hs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B login successful


In [10]:
dataset = load_dataset(DATASET_NAME)

print(dataset)

README.md:   0%|          | 0.00/509 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.31MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/val-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  216kB            

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 20000
    })
    val: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1000
    })
})


In [11]:
train = dataset["train"]

val = dataset["val"]

if len(val) > VAL_SIZE:
    val = val.select(range(VAL_SIZE))

test = dataset["test"]

print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))

Train: 20000
Validation: 500
Test: 1000


In [12]:
print(train.column_names)
print(train[0])

['prompt', 'completion']
{'prompt': 'What does this cost to the nearest dollar?\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.\n\nPrice is $', 'completion': '64.00'}


In [13]:
# Optional

if len(train) > 10000:
    train = train.select(range(10000))

print("Training examples:", len(train))

Training examples: 10000


In [14]:
if QUANT_4_BIT:

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=(
            torch.bfloat16
            if use_bf16
            else torch.float16
        ),
        bnb_4bit_quant_type="nf4",
    )

else:

    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=(
            torch.bfloat16
            if use_bf16
            else torch.float16
        ),
    )

print("✅ Quantization configured")

✅ Quantization configured


In [15]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded")
print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Tokenizer loaded
Pad token: <|endoftext|>


In [16]:
use_bf16 = False

print("FP16: False")
print("BF16: False")

FP16: False
BF16: False


In [17]:
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,

    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=10,
    weight_decay=WEIGHT_DECAY,
    optim=OPTIMIZER,

    # IMPORTANT: disable AMP completely
    fp16=False,
    bf16=False,

    max_grad_norm=0.3,
    gradient_checkpointing=True,

    logging_steps=LOG_STEPS,

    eval_strategy="steps",
    eval_steps=SAVE_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,

    report_to="none",

    push_to_hub=False,
)

print("✅ FP16/BF16 disabled")

✅ FP16/BF16 disabled


In [18]:
from peft import LoraConfig

# LoRA configuration.
# LoRA trains a small number of adapter parameters instead of the entire model.
lora_parameters = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("LoRA configuration created successfully.")

LoRA configuration created successfully.


In [19]:
!pip install -U "torchao>=0.16.0" peft trl transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [20]:
fine_tuning = SFTTrainer(
    model=base_model,
    args=train_parameters,
    train_dataset=train,
    eval_dataset=val,
    processing_class=tokenizer,
    peft_config=lora_parameters,
)

print("✅ Trainer created")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Adding EOS to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Trainer created


In [21]:
for name, param in fine_tuning.model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype, param.device)
        break

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.float32 cuda:0


In [22]:
training_result = fine_tuning.train()

print("✅ Training complete")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.858416,0.840625,0.824503,349172.000000,0.705513
200,0.807311,0.822885,0.813339,698042.000000,0.712778
300,0.801416,0.817076,0.828076,1048950.000000,0.712402
313,0.827512,0.817042,0.827834,1092875.000000,0.712402


✅ Training complete


In [23]:
eval_results = fine_tuning.evaluate()

print(eval_results)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.827512,0.817042,313,0.827834,1092875.000000,0.712402


{'eval_loss': 0.8170419335365295, 'eval_entropy': 0.8278342059680394, 'eval_num_tokens': 1092875.0, 'eval_mean_token_accuracy': 0.7124020873554169}


In [24]:
fine_tuning.save_model(PROJECT_RUN_NAME)

tokenizer.save_pretrained(PROJECT_RUN_NAME)

print("✅ Model saved locally")

✅ Model saved locally


In [25]:
push_to_hub=True

In [26]:
fine_tuning.push_to_hub()

print("✅ Uploaded to Hugging Face")
print("Model:", HUB_MODEL_NAME)

✅ Uploaded to Hugging Face
Model: Charan2804/price-2026-08-25_08.25.57-lite


In [27]:
if LOG_TO_WANDB:
    wandb.finish()

print("✅ Training run finished")

✅ Training run finished
